In [19]:
import pandas as pd


from utils.participant_data import load_participant
from utils.catchtrials import CatchTrials
from utils.gazedata import GazeData

In [20]:
participant_nr = "00"

DATA_ROOT = "data"
PARTICIPANT_FOLDER_PATTERN = f"participant_{participant_nr}"

df_pos = pd.read_csv("data/sentence_position.csv")

In [ ]:
from utils.pres_mapper import PresentationMapper

mapper = PresentationMapper(
    real_order_path="data/sentences.js",
    exp_order_path=f"{DATA_ROOT}/{PARTICIPANT_FOLDER_PATTERN}/sentence_order_{participant_nr}",
)

df_pos_mapped = df_pos.copy()
df_pos_mapped["presentation_index"] = df_pos_mapped["real_index"].map(
    mapper.get_real_to_pres_map()
)

assert df_pos_mapped["presentation_index"].notna().all()

# Catch trials analysis

In [21]:

p = load_participant(participant_nr)   # or whatever participant_nr you're on

p.calibration        # dict from the JSON
p.catch_trials        # DataFrame
p.experiment           # DataFrame
p.gaze                 # DataFrame
p.sentence_order       # DataFrame

CatchTrials.from_dataframe(p.catch_trials).summary()  # reuse your existing analysis on just this one

Total trials     : 10
Unique sentences : 9
Percent correct  : 90.0%
Mean RT          : 4212.5 ms

RT by correctness:
           count     min      q1  median      q3     max
response                                                
incorrect    1.0  5296.0  5296.0  5296.0  5296.0  5296.0
correct      9.0  1776.0  2976.0  3904.0  4032.0  8303.0
Catch types      : 
catch_type
seen      6
unseen    4
Name: count, dtype: int64


# Gaze analysis

In [22]:
print(f"Shape: {p.gaze.shape}")
print(f"\nColumns: {list(p.gaze.columns)}")
print(f"\nDtypes:\n{p.gaze.dtypes}")

Shape: (55175, 1)

Columns: ['CNT\tTIME\tTIME_TICK\tFPOGX\tFPOGY\tFPOGS\tFPOGD\tFPOGID\tFPOGV\tLPOGX\tLPOGY\tLPOGV\tRPOGX\tRPOGY\tRPOGV\tBPOGX\tBPOGY\tBPOGV\tLPCX\tLPCY\tLPD\tLPS\tLPV\tRPCX\tRPCY\tRPD\tRPS\tRPV\tLEYEX\tLEYEY\tLEYEZ\tLPUPILD\tLPUPILV\tREYEX\tREYEY\tREYEZ\tRPUPILD\tRPUPILV\tCX\tCY\tCS\tUSER']

Dtypes:
CNT\tTIME\tTIME_TICK\tFPOGX\tFPOGY\tFPOGS\tFPOGD\tFPOGID\tFPOGV\tLPOGX\tLPOGY\tLPOGV\tRPOGX\tRPOGY\tRPOGV\tBPOGX\tBPOGY\tBPOGV\tLPCX\tLPCY\tLPD\tLPS\tLPV\tRPCX\tRPCY\tRPD\tRPS\tRPV\tLEYEX\tLEYEY\tLEYEZ\tLPUPILD\tLPUPILV\tREYEX\tREYEY\tREYEZ\tRPUPILD\tRPUPILV\tCX\tCY\tCS\tUSER    str
dtype: object


In [23]:
path = f"{DATA_ROOT}/{PARTICIPANT_FOLDER_PATTERN}/gaze_{participant_nr}.csv"

gaze = GazeData(path)


## simple statistics

In [24]:
print("Fixation Statistics:")
print(gaze.fixation_stats())
print("\nPupil Statistics:")
print(gaze.pupil_stats())

Fixation Statistics:
n_fixations        2106.00
mean_duration         0.25
median_duration       0.20
mean_x                0.49
mean_y                0.46
dtype: float64

Pupil Statistics:
             left     right
mean        15.78     15.47
median      15.51     15.21
n_valid  45912.00  46100.00


In [25]:
gaze.summary()

Total samples    : 55175
Duration         : 10497.49 s
Valid fixations  : 37283
User events      : 55175

Fixation stats:
n_fixations        2106.00
mean_duration         0.25
median_duration       0.20
mean_x                0.49
mean_y                0.46
dtype: float64

Pupil stats:
             left     right
mean        15.78     15.47
median      15.51     15.21
n_valid  45912.00  46100.00


In [26]:
print("sentence duration")
gaze.sentence_duration_stats()

sentence duration


mean       3.46
median     2.77
min        0.00
max       15.88
std        2.81
dtype: float64

## gaze plots

In [27]:
presentation = 13

results = gaze.gaze_on_screen(df_pos)

gaze.plot_gaze_on_text(df_pos, pres=presentation, results=results)

sacc = gaze.saccades_movement(pres=presentation, df_position=df_pos)
sacc["is_regression"].mean()          # overall regression rate
sacc["saccade_type"].value_counts()   # breakdown: line vs within-line regressions

gaze.plot_saccades_on_text(df_pos, pres=presentation, saccades=sacc, show_gaze=True, results=results)

KeyError: 'presentation_index'